In [1]:
import pandas as pd

In [ ]:
import pandas as pd

dsc_xl = pd.ExcelFile('/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/DSC/2026-07-17/20260717-DSC-UAFW.xls')
dsc_df = pd.read_excel('/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/DSC/2026-07-17/20260717-DSC-UAFW.xls', 
                          sheet_name="Ramp 10.00 °Cmin to 30.00 °C",
                          header =1
                          )
print(dsc_xl.sheet_names)
dsc_df

# extract data fxn

In [ ]:

# if you want a df of the details, use:
# details_df = pd.read_excel("path_to_excel_file", sheet_name = "Details", header = 1)

def extract_data(path): 
    # Read in original file 
    xl = pd.ExcelFile(path) 
    all_sheets = xl.sheet_names 
    print(all_sheets)
    
    data_sheets = all_sheets[1:] 
    df_list = [] 
    
    for sheet in data_sheets: 
        sheet_df = pd.read_excel(path, sheet_name=sheet, header=1) 
        
        # extract the units from the first row of data (index 0)
        units = sheet_df.iloc[0].fillna('').astype(str).tolist()
        
        # combine old column names with the units
        new_columns = []
        for col, unit in zip(sheet_df.columns, units):
            clean_col = col.split('.')[0] if '.' in col else col
            if unit:
                new_columns.append(f"{clean_col} ({unit})")
            else:
                new_columns.append(clean_col)
                
        # assign the new combined names back to the dataframe columns
        sheet_df.columns = new_columns
        
        # drop units row from the data
        sheet_df = sheet_df.drop(index=0).reset_index(drop=True)
        
        # add source sheet column
        sheet_df['Source Sheet'] = sheet 
        df_list.append(sheet_df) 
        
    # Vertically stack all the sheets together into one final df 
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # convert to float
    # errors='coerce' turns any unconvertible text or bad data into NaN safely
    for col in combined_df.columns:
        if col != 'Source Sheet':
            combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')
    
    # make sure source sheet is last column in final df
    cols = [col for col in combined_df.columns if col != 'Source Sheet'] + ['Source Sheet']
    combined_df = combined_df[cols]
    
    return combined_df

In [ ]:
dsc_df = extract_data('/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/TGA/2026-07-13/2026-07-13-TGA-UAFW.xls')
dsc_df.head()

In [ ]:
import matplotlib.pyplot as plt

dsc_cols = list(dsc_df.columns)

y1 = "indigo"
y2 = "teal"

fig, ax1 = plt.subplots()

heatflow_Wg = ax1.plot(dsc_df[dsc_cols[1]], dsc_df[dsc_cols[3]], color = y1, lw = 2, label = dsc_cols[3])
ax1.set_xlabel(dsc_cols[1])
ax1.set_ylabel(dsc_cols[3])
ax1.tick_params(axis='y')
ax1.set_xlim(0,400)

ax2 = ax1.twinx()

heatflow_JgC = ax2.plot(dsc_df[dsc_cols[1]], dsc_df[dsc_cols[4]], color = y2, lw = 2, label = dsc_cols[4])
ax2.set_ylabel(dsc_cols[4])
ax2.tick_params(axis = 'y')

plt.title("TGA UAFW 1:2:4 (2026-07-08)", fontsize=15)
plt.legend(loc = "center right", handles = [heatflow_Wg[0], 
                                            heatflow_JgC[0]
                                            ])
plt.tight_layout()
plt.show()

# plotting

In [ ]:
import matplotlib.pyplot as plt

dsc_cols = list(dsc_df.columns)

y1 = "magenta"
y2 = "orange"

fig, (ax1, ax2) = plt.subplots(1 ,2, figsize = (12,5))

heatflow_Wg = ax1.plot(dsc_df[dsc_cols[1]], dsc_df[dsc_cols[3]], color = y1, lw = 2, label = dsc_cols[3])
ax1.set_xlabel(dsc_cols[1])
ax1.set_ylabel(dsc_cols[3])
ax1.tick_params(axis='y')
# ax1.set_xlim(20,500)

ax1_twin = ax1.twinx()

heatflow_JgC = ax1_twin.plot(dsc_df[dsc_cols[1]], dsc_df[dsc_cols[3]], color = y2, lw = 2, label = dsc_cols[3])
ax1_twin.set_ylabel(dsc_cols[3])
ax1_twin.tick_params(axis = 'y')
ax1.legend(loc = "center right", handles = [heatflow_Wg[0], 
                                            heatflow_JgC[0],]
                                            )

# subplot 2 ****** NEED TO CHANGE ***********
heatflow_Wg2 = ax2.plot(dsc_df[dsc_cols[1]], dsc_df[dsc_cols[3]], color = y1, lw = 2, label = dsc_cols[3])
ax2.set_xlabel(dsc_cols[1])
ax2.set_ylabel(dsc_cols[3])
ax2.tick_params(axis='y')
ax2.set_xlim(20,200)

ax2_twin = ax2.twinx()

heatflow_JgC2 = ax2_twin.plot(dsc_df[dsc_cols[1]], dsc_df[dsc_cols[4]], color = y2, lw = 2, label = dsc_cols[4])
ax2_twin.set_ylabel(dsc_cols[4])
ax2_twin.tick_params(axis = 'y')
ax2.legend(loc = "lower left", handles = [heatflow_Wg2[0], 
                                            heatflow_JgC2[0]]
                                            )

plt.suptitle("DSC UAFW 1:2:4 (2026-07-17)", fontsize=15)
# plt.legend()
plt.tight_layout()
plt.show()
